# 03 · Modeling: champion scorecard vs challenger LightGBM

**Owner:** Credit Risk Analyst · **Audience:** Head of Credit Policy, Model Validation · **Inputs:** `fct_application_features` (notebook 02)

**Layout:** the first code cells fit and score everything; the TL;DR is generated from those results; the sections
after it show the evidence. Headline numbers are on the out-of-time test window (Jan to Mar 2020).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import pickle

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression

from src import charts
from src.config import artifact, load_config, processed
from src.db import connect
from src.economics import Economics
from src.explain import global_importance, shap_values, top_reasons
from src.features import CATEGORICAL, FEATURES, GROUPS, LABELS, NUMERIC
from src.metrics import auc, brier, calibration_table, gini, ks
from src.modeling import PlattCalibrator, fit_scorecard, prepare_lgbm_frame, scorecard_reason_codes, train_lgbm
from src.strategy import approve_at_rate, approve_ev_positive, evaluate_policy

pio.renderers.default = "notebook+png"
pd.set_option("display.max_colwidth", 120)
cfg = load_config()
SEED = cfg["seed"]
econ = Economics.from_config(cfg)
OUT = processed(cfg, "dashboard")
OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

con = connect(cfg, read_only=True)
df = con.execute("SELECT * FROM fct_application_features ORDER BY case_id").df()
con.close()
for c in CATEGORICAL:
    df[c] = df[c].astype(object)
y = df.target.to_numpy(int)
M = {s: (df.split == s).to_numpy() for s in ["train", "valid", "test", "monitor"]}
gain, loss = econ.per_loan(df.loan_amount, df.product_type)
print({s: int(m.sum()) for s, m in M.items()})

In [ ]:
# Champion: WOE logistic-regression scorecard (optbinning, monotonic bins, IV + correlation selection, sign check)
sc = fit_scorecard(df.loc[M["train"], FEATURES], y[M["train"]], NUMERIC, CATEGORICAL, max_features=18, seed=SEED)
raw_champ = sc.scorecard.predict_proba(df[sc.selected])[:, 1]
points = sc.scorecard.score(df[sc.selected])
cal_champ = PlattCalibrator().fit(raw_champ[M["valid"]], y[M["valid"]])
pd_champ = cal_champ.transform(raw_champ)
print("scorecard variables:", len(sc.selected))

In [ ]:
# Challenger: LightGBM on all 90 point-in-time features, early stopping on the validation window
Xtr, cat_levels = prepare_lgbm_frame(df[M["train"]], FEATURES, CATEGORICAL)
Xva, _ = prepare_lgbm_frame(df[M["valid"]], FEATURES, CATEGORICAL, cat_levels)
booster = train_lgbm(Xtr, y[M["train"]], Xva, y[M["valid"]], seed=SEED)
X_all, _ = prepare_lgbm_frame(df, FEATURES, CATEGORICAL, cat_levels)
raw_chal = booster.predict(X_all)
cal_chal = PlattCalibrator().fit(raw_chal[M["valid"]], y[M["valid"]])
pd_chal = cal_chal.transform(raw_chal)
print("boosting rounds:", booster.best_iteration)

In [ ]:
# Fairness experiment: the same challenger plus grouped employment status (contains a 'retired' group, a proxy for age)
emp_map = {"PRIVATE_SECTOR_EMPLOYEE": "employed", "SALARIED_GOVT": "employed", "EMPLOYED": "employed", "SELFEMPLOYED": "self-employed", "RETIRED_PENSIONER": "retired"}
df["employment_status"] = df.fair_income_type.map(emp_map).fillna("other").astype(object)
feats_v, cats_v = FEATURES + ["employment_status"], CATEGORICAL + ["employment_status"]
Xtr_v, lv_v = prepare_lgbm_frame(df[M["train"]], feats_v, cats_v)
Xva_v, _ = prepare_lgbm_frame(df[M["valid"]], feats_v, cats_v, lv_v)
booster_v = train_lgbm(Xtr_v, y[M["train"]], Xva_v, y[M["valid"]], seed=SEED)
raw_v = booster_v.predict(prepare_lgbm_frame(df, feats_v, cats_v, lv_v)[0])
pd_chal_emp = PlattCalibrator().fit(raw_v[M["valid"]], y[M["valid"]]).transform(raw_v)

In [ ]:
# Imbalance handling experiment (appendix): does re-weighting or SMOTE help the decision?
from imblearn.over_sampling import SMOTE

te, va = M["test"], M["valid"]
pos_w = (y[M["train"]] == 0).sum() / (y[M["train"]] == 1).sum()
booster_bal = train_lgbm(Xtr, y[M["train"]], Xva, y[M["valid"]], params={"scale_pos_weight": pos_w}, seed=SEED)
raw_bal = booster_bal.predict(X_all)

woe = sc.scorecard.binning_process_.transform(df[sc.selected], metric="woe")
Wtr, Wte = woe[M["train"]].to_numpy(), woe[te].to_numpy()
lr_plain = LogisticRegression(max_iter=2000).fit(Wtr, y[M["train"]])
lr_bal = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Wtr, y[M["train"]])
rng = np.random.default_rng(SEED)
sub = rng.choice(np.nonzero(M["train"])[0], size=300_000, replace=False)
Xs, ys = SMOTE(random_state=SEED).fit_resample(woe.iloc[sub].to_numpy(), y[sub])
lr_smote = LogisticRegression(max_iter=2000).fit(Xs, ys)

def decision_outcome(p, mask):
    res = evaluate_policy(approve_ev_positive(p[mask], gain[mask], loss[mask]), y[mask], gain[mask], loss[mask])
    return res["approval_rate"], res["net_profit"]

imb_rows = []
for name, p_all in [
    ("LightGBM, no re-weighting", raw_chal),
    ("LightGBM, class weight (balanced)", raw_bal),
    ("Logistic on WOE, no re-weighting", lr_plain.predict_proba(woe.to_numpy())[:, 1]),
    ("Logistic on WOE, class weight (balanced)", lr_bal.predict_proba(woe.to_numpy())[:, 1]),
    ("Logistic on WOE, SMOTE (300k sample)", lr_smote.predict_proba(woe.to_numpy())[:, 1]),
]:
    p_cal = PlattCalibrator().fit(p_all[va], y[va]).transform(p_all)
    appr_raw, prof_raw = decision_outcome(p_all, te)
    appr_cal, prof_cal = decision_outcome(p_cal, te)
    imb_rows.append({
        "approach": name, "test_auc": auc(y[te], p_all[te]), "test_brier_raw": brier(y[te], p_all[te]),
        "mean_pd_raw": p_all[te].mean(), "observed_bad_rate": y[te].mean(),
        "approval_rate_if_cutoff_on_raw_output": appr_raw, "net_profit_if_cutoff_on_raw_output": prof_raw,
        "approval_rate_after_recalibration": appr_cal, "net_profit_after_recalibration": prof_cal,
    })
imbalance = pd.DataFrame(imb_rows)
imbalance.to_parquet(OUT / "imbalance_experiment.parquet", index=False)

In [ ]:
# Rank-ordering and calibration metrics by window (secondary metrics for the model card)
rows = []
for model, p in [("Champion (scorecard)", pd_champ), ("Challenger (LightGBM)", pd_chal)]:
    for s in ["train", "valid", "test", "monitor"]:
        m = M[s]
        rows.append({"model": model, "window": s, "loans": int(m.sum()), "bad_rate": y[m].mean(), "mean_pd": p[m].mean(),
                     "calibration_ratio": p[m].mean() / y[m].mean(), "auc": auc(y[m], p[m]), "gini": gini(y[m], p[m]), "ks": ks(y[m], p[m]), "brier": brier(y[m], p[m])})
model_metrics = pd.DataFrame(rows)
model_metrics.to_parquet(OUT / "model_metrics.parquet", index=False)

calib = pd.concat([calibration_table(y[te], p[te], 10).assign(model=name) for name, p in
                   [("Champion (scorecard)", pd_champ), ("Challenger (LightGBM)", pd_chal)]], ignore_index=True)
calib.to_parquet(OUT / "calibration_test.parquet", index=False)

monthly = []
for name, p in [("Champion (scorecard)", pd_champ), ("Challenger (LightGBM)", pd_chal)]:
    g = pd.DataFrame({"m": df.decision_month, "y": y, "p": p}).groupby("m").agg(bad_rate=("y", "mean"), mean_pd=("p", "mean"), n=("y", "size")).reset_index()
    g["model"] = name
    monthly.append(g)
monthly = pd.concat(monthly, ignore_index=True).rename(columns={"m": "decision_month"})

In [ ]:
# Money view on the test window: the cost-based rule and fixed approval rates
biz = []
for name, p in [("Champion (scorecard)", pd_champ), ("Challenger (LightGBM)", pd_chal)]:
    pt = p[te]
    pol = {"Cost-based rule (approve if expected profit > 0)": approve_ev_positive(pt, gain[te], loss[te])}
    for r in [0.90, 0.95]:
        pol[f"Approve the safest {r:.0%}"] = approve_at_rate(pt, r)
    for pname, a in pol.items():
        biz.append({"model": name, "policy": pname, **evaluate_policy(a, y[te], gain[te], loss[te], df.loan_amount.to_numpy()[te])})
biz = pd.DataFrame(biz)
biz.to_parquet(OUT / "model_business_comparison.parquet", index=False)

In [ ]:
# Segments: new-to-lender vs existing clients, and by product (test window)
seg_rows = []
for seg_name, seg in [("New to lender", df.is_new_to_lender == 1), ("Existing client", df.is_new_to_lender == 0),
                      ("Cash loan", df.product_type == "CAL"), ("Point-of-sale loan", df.product_type == "COL"), ("Revolving", df.product_type == "REL")]:
    m = te & seg.to_numpy()
    for name, p in [("Champion (scorecard)", pd_champ), ("Challenger (LightGBM)", pd_chal)]:
        seg_rows.append({"segment": seg_name, "model": name, "loans": int(m.sum()), "share_of_test": m.sum() / te.sum(), "bad_rate": y[m].mean(),
                         "mean_pd": p[m].mean(), "calibration_ratio": p[m].mean() / y[m].mean(), "gini": gini(y[m], p[m])})
segments = pd.DataFrame(seg_rows)
segments.to_parquet(OUT / "segment_metrics.parquet", index=False)

In [ ]:
# Explanations: SHAP for the challenger (global + a sample of individual decisions), points lost for the scorecard
test_idx = np.nonzero(te)[0]
g_idx = rng.choice(test_idx, size=20_000, replace=False)
shap_g, base_value = shap_values(booster, X_all.iloc[g_idx])
imp = global_importance(shap_g, LABELS, GROUPS)
imp.to_parquet(OUT / "shap_global.parquet", index=False)

declined = test_idx[~approve_ev_positive(pd_chal[te], gain[te], loss[te])]
approved = test_idx[approve_ev_positive(pd_chal[te], gain[te], loss[te])]
s_idx = np.sort(np.concatenate([rng.choice(declined, size=min(1500, len(declined)), replace=False), rng.choice(approved, size=500, replace=False)]))
shap_s, _ = shap_values(booster, X_all.iloc[s_idx])
reasons = top_reasons(shap_s, LABELS, k=4)
sc_reasons = scorecard_reason_codes(sc, df.iloc[s_idx][sc.selected], LABELS, top_k=4).add_prefix("sc_")
sample = df.iloc[s_idx][["case_id", "decision_month", "target", "product_type", "loan_amount", "monthly_payment", "income_main"]].copy()
sample["pd_challenger"], sample["pd_champion"], sample["score_champion"] = pd_chal[s_idx], pd_champ[s_idx], points[s_idx]
sample["breakeven_pd"] = econ.breakeven_per_loan(sample.product_type)
sample["decision_challenger"] = np.where(sample.pd_challenger < sample.breakeven_pd, "Approve", "Decline")
sample["decision_champion"] = np.where(sample.pd_champion < sample.breakeven_pd, "Approve", "Decline")
sample = pd.concat([sample.reset_index(drop=True), reasons.reset_index(drop=True), sc_reasons.reset_index(drop=True)], axis=1)
sample.to_parquet(OUT / "explain_sample.parquet", index=False)
shap_s.reset_index(drop=True).assign(case_id=sample.case_id.values).to_parquet(OUT / "explain_shap.parquet", index=False)
fv = df.iloc[s_idx][FEATURES].reset_index(drop=True).astype(object)
fv.insert(0, "case_id", sample.case_id.values)
fv.astype(str).to_parquet(OUT / "explain_values.parquet", index=False)  # display values
(OUT / "explain_meta.json").write_text(json.dumps({"base_value_logodds": base_value, "calibration_a": cal_chal.a, "calibration_b": cal_chal.b}))

# typical (median) values among approved test applicants, for "your value vs typical" in the dashboard
typical = df.iloc[approved][NUMERIC].median().rename("typical_approved").reset_index().rename(columns={"index": "feature"})
typical.to_parquet(OUT / "typical_values.parquet", index=False)

# scorecard documentation tables
sc.points_table.to_parquet(OUT / "scorecard_points.parquet", index=False)
sc.selection_log.to_parquet(OUT / "scorecard_selection.parquet", index=False)
coefs = pd.DataFrame({"feature": sc.selected, "label": [LABELS[f] for f in sc.selected], "coefficient": sc.scorecard.estimator_.coef_[0],
                      "iv": [float(sc.screening.set_index("name").loc[f, "iv"]) for f in sc.selected]})
coefs.to_parquet(OUT / "scorecard_coefficients.parquet", index=False)

In [ ]:
# Persist predictions for notebook 04, the dashboard and the report
pred = df[["case_id", "split", "decision_month", "target", "product_type", "loan_amount", "monthly_payment", "income_main", "is_new_to_lender",
           "fair_sex", "fair_age_years", "fair_income_type"]].copy()
pred["pd_champion"], pred["pd_challenger"], pred["pd_challenger_emp"], pred["score_champion"] = pd_champ, pd_chal, pd_chal_emp, points
pred["pd_champion_raw"], pred["pd_challenger_raw"] = raw_champ, raw_chal
pred.to_parquet(processed(cfg, "predictions.parquet"), index=False)
booster.save_model(str(artifact(cfg, "challenger_lightgbm.txt")), num_iteration=booster.best_iteration)
with open(artifact(cfg, "champion_scorecard.pkl"), "wb") as f:
    pickle.dump(sc.scorecard, f)
(artifact(cfg, "calibration.json")).write_text(json.dumps({"champion": [cal_champ.a, cal_champ.b], "challenger": [cal_chal.a, cal_chal.b]}))
# monitoring inputs: top challenger features by SHAP, all windows (for CSI in notebook 04)
mon_cols = ["case_id", "decision_month", "split"] + imp.feature.head(12).tolist()
df[mon_cols].to_parquet(processed(cfg, "monitoring_features.parquet"), index=False)

In [ ]:
grp_share = imp.groupby("group").share.sum().sort_values(ascending=False)
track = float(grp_share.get("Repayment history", 0) + grp_share.get("Previous applications", 0))
mm = model_metrics.set_index(["model", "window"])
g_ch_te, g_cp_te = mm.loc[("Challenger (LightGBM)", "test"), "gini"], mm.loc[("Champion (scorecard)", "test"), "gini"]
g_ch_va, g_cp_va = mm.loc[("Challenger (LightGBM)", "valid"), "gini"], mm.loc[("Champion (scorecard)", "valid"), "gini"]
cr_ch_te, cr_cp_te = mm.loc[("Challenger (LightGBM)", "test"), "calibration_ratio"], mm.loc[("Champion (scorecard)", "test"), "calibration_ratio"]
bz = biz.set_index(["model", "policy"])
rule = "Cost-based rule (approve if expected profit > 0)"
ch, cp = bz.loc[("Challenger (LightGBM)", rule)], bz.loc[("Champion (scorecard)", rule)]
p95 = "Approve the safest 95%"
seg = segments.set_index(["segment", "model"])
summary = {
    "scorecard_variables": len(sc.selected), "lgbm_rounds": int(booster.best_iteration),
    "gini_test_challenger": g_ch_te, "gini_test_champion": g_cp_te, "gini_valid_challenger": g_ch_va, "gini_valid_champion": g_cp_va,
    "auc_test_challenger": mm.loc[("Challenger (LightGBM)", "test"), "auc"], "auc_test_champion": mm.loc[("Champion (scorecard)", "test"), "auc"],
    "ks_test_challenger": mm.loc[("Challenger (LightGBM)", "test"), "ks"], "ks_test_champion": mm.loc[("Champion (scorecard)", "test"), "ks"],
    "calibration_ratio_test_challenger": cr_ch_te, "calibration_ratio_test_champion": cr_cp_te,
    "bad_rate_valid": float(y[va].mean()), "bad_rate_test": float(y[te].mean()),
    "rule_approval_challenger": ch.approval_rate, "rule_approval_champion": cp.approval_rate,
    "rule_bad_rate_challenger": ch.bad_rate_approved, "rule_bad_rate_champion": cp.bad_rate_approved,
    "rule_profit_challenger": ch.net_profit, "rule_profit_champion": cp.net_profit,
    "p95_bad_rate_challenger": bz.loc[("Challenger (LightGBM)", p95), "bad_rate_approved"], "p95_bad_rate_champion": bz.loc[("Champion (scorecard)", p95), "bad_rate_approved"],
    "p95_loss_challenger": bz.loc[("Challenger (LightGBM)", p95), "credit_loss"], "p95_loss_champion": bz.loc[("Champion (scorecard)", p95), "credit_loss"],
    "gini_test_new_challenger": seg.loc[("New to lender", "Challenger (LightGBM)"), "gini"], "gini_test_existing_challenger": seg.loc[("Existing client", "Challenger (LightGBM)"), "gini"],
    "top_driver": imp.label[0], "top_driver_group": imp.group[0], "track_record_share": track, "group_shares": grp_share.to_dict(),
}
(processed(cfg, "nb03_summary.json")).write_text(json.dumps(summary, indent=2, default=float))
display(Markdown(f"""
## TL;DR

- **The challenger ranks risk better than the scorecard on every window.** Out-of-time test Gini **{g_ch_te:.3f}** (LightGBM) vs **{g_cp_te:.3f}** (18-variable
  scorecard); validation {g_ch_va:.3f} vs {g_cp_va:.3f}. AUC is reported for the model card only; the decision is made on money below.
- **In money (test window, same cost-based rule):** the challenger approves **{ch.approval_rate:.1%}** of applications with a **{ch.bad_rate_approved:.2%}**
  bad rate and **{charts.fmt_money(ch.net_profit)}** net profit, vs **{cp.approval_rate:.1%}**, **{cp.bad_rate_approved:.2%}** and **{charts.fmt_money(cp.net_profit)}** for the champion.
  At the same 95% approval rate the challenger's book has a **{bz.loc[('Challenger (LightGBM)', p95), 'bad_rate_approved']:.2%}** bad rate vs
  **{bz.loc[('Champion (scorecard)', p95), 'bad_rate_approved']:.2%}**.
- **Calibration drift:** both models were calibrated on Nov to Dec 2019 (bad rate {y[va].mean():.2%}); the test quarter defaulted at {y[te].mean():.2%}, so predicted PDs
  are too low by about {1 - cr_ch_te:.0%} (calibration ratio {cr_ch_te:.2f}). Rank-ordering held up; the level did not. The cut-off section in notebook 04 handles this.
- **Why are people declined?** The client's track record with the lender (repayment history and previous applications) carries **{track:.0%}** of the
  challenger's total impact; the strongest single inputs are *{imp.label[0].lower()}*, *{imp.label[1].lower()}* and *{imp.label[2].lower()}*. Every decline gets
  up to four plain-English reasons from SHAP (challenger) or points lost (scorecard).
- **No resampling.** Class weights and SMOTE leave the ranking about the same but break the probability scale; applied naively to the cost-based cut-off they
  decline far too many good customers (see the appendix).
"""))

## 1. Question

Is a gradient-boosted model worth adopting over the logistic-regression scorecard that a lender like this typically
runs, once both are judged on the money they make or save at a realistic cut-off, not on AUC? And can its decisions be
explained to an applicant and to a regulator?

## 2. Method

| Step | Champion (scorecard) | Challenger (LightGBM) |
|---|---|---|
| Inputs | the same 90 point-in-time features (notebook 02) | same |
| Transformation | monotonic optimal binning, weight of evidence (WOE) | none (native missing values and categories) |
| Variable selection | IV ≥ 0.02, WOE correlation ≤ 0.6, max 18 variables, all coefficients must have the expected sign | all 90; regularised by leaf size, L2 and early stopping |
| Fit window | train (Jan to Oct 2019) | train; early stopping on validation (Nov to Dec 2019) |
| Output | points (600 = odds 50:1, +20 points doubles the odds) and PD | PD |
| Calibration | Platt scaling on validation | Platt scaling on validation |

Class imbalance (3% bads) is handled by **not** resampling and choosing the cut-off from costs instead; the appendix shows why.

## 3. Results
### 3.1 The champion scorecard

In [ ]:
display(coefs.sort_values("iv", ascending=False).reset_index(drop=True))
pts = sc.points_table[["Variable", "Bin", "Count", "Event rate", "WoE", "Points"]].copy()
pts["Variable"] = pts.Variable.map(LABELS)
pts.head(30)

Each variable's bins carry points; an applicant's score is the sum. Bins are forced to be monotonic, so the points move
in one direction (e.g. more days past due never earns more points), which is what makes the scorecard easy to explain
and to challenge. The selection log in `data/processed/dashboard/scorecard_selection.parquet` records why each candidate
was or was not used.

### 3.2 Rank-ordering (model-card metrics)

In [ ]:
model_metrics.pivot_table(index="window", columns="model", values=["gini", "ks", "calibration_ratio"]).reindex(["train", "valid", "test", "monitor"]).round(3)

### 3.3 Calibration on the out-of-time test window

In [ ]:
fig = go.Figure()
lim = float(max(calib.mean_pd.max(), calib.bad_rate.max())) * 1.08
fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode="lines", line=dict(color=charts.AXIS, width=1), name="Perfect calibration", hoverinfo="skip"))
for name in ["Champion (scorecard)", "Challenger (LightGBM)"]:
    c = calib[calib.model == name]
    fig.add_trace(go.Scatter(x=c.mean_pd, y=c.bad_rate, mode="lines+markers", name=name, line=dict(color=charts.MODEL_COLORS[name], width=2),
                             marker=dict(size=8, line=dict(color=charts.SURFACE, width=2)),
                             error_y=dict(type="data", symmetric=False, array=c.ci_high - c.bad_rate, arrayminus=c.bad_rate - c.ci_low, color=charts.MODEL_COLORS[name], thickness=1, width=0),
                             hovertemplate="predicted %{x:.2%}<br>observed %{y:.2%}<extra>" + name + "</extra>"))
charts.base_layout(fig, "Both models rank well but under-predict the test quarter's higher default level",
                   "Observed bad rate vs mean predicted PD by PD decile, Jan to Mar 2020 (95% intervals)", height=460)
fig.update_xaxes(title="Mean predicted PD", tickformat=".0%", range=[0, lim])
fig.update_yaxes(title="Observed bad rate", tickformat=".0%", range=[0, lim])
fig

In [ ]:
fig = go.Figure()
obs = monthly[monthly.model == "Challenger (LightGBM)"]
x = [charts._month_label(m) for m in obs.decision_month]
fig.add_trace(go.Scatter(x=x, y=obs.bad_rate, name="Observed bad rate", mode="lines+markers", line=dict(color=charts.INK_2, width=2), marker=dict(size=7)))
for name in ["Champion (scorecard)", "Challenger (LightGBM)"]:
    g = monthly[monthly.model == name]
    fig.add_trace(go.Scatter(x=x, y=g.mean_pd, name=f"Mean PD, {name}", mode="lines", line=dict(color=charts.MODEL_COLORS[name], width=2)))
charts.base_layout(fig, "Predicted PD tracks the trend but misses the early-2020 rise and the COVID-period drop",
                   "Mean predicted PD vs observed bad rate by decision month (%)", height=420)
fig.update_yaxes(tickformat=".1%", rangemode="tozero")
fig

Calibration-in-the-large moves with the population: the models were calibrated on a 3.5% quarter and tested on a 4.4%
quarter. This is not something a better model fixes; it is why the monitoring plan tracks the calibration ratio monthly
and why notebook 04 checks the cut-off still holds if the base rate rises by a quarter.

### 3.4 What it means in money (test window)

In [ ]:
view = biz[["model", "policy", "approval_rate", "bad_rate_approved", "capture_rate", "false_positive_rate", "credit_loss", "forgone_margin", "net_profit"]].copy()
view

### 3.5 Segments: new-to-lender vs existing clients, and products

In [ ]:
segments.pivot_table(index="segment", columns="model", values=["gini", "calibration_ratio"]).round(3)

New-to-lender clients have no internal repayment history, so both models lean on the application and bureau data and
rank them less well. A segmented scorecard (or segment-specific cut-offs) is a standard next step; it is listed in the
recommendations.

### 3.6 Why does the model decline someone? (explainability)

In [ ]:
top = imp.head(20).iloc[::-1]
fig = go.Figure(go.Bar(x=top.mean_abs_shap, y=top.label, orientation="h", marker_color=charts.BLUE, width=0.6,
                       customdata=np.c_[top.feature, top.group], hovertemplate="%{y}<br>%{customdata[0]} · %{customdata[1]}<br>mean |SHAP| %{x:.3f}<extra></extra>"))
charts.base_layout(fig, f"The client's track record carries {track:.0%} of the challenger's impact; no single input dominates",
                   "Average absolute impact on the log-odds of default (mean |SHAP|), 20,000 test applicants, top 20 inputs", height=560, legend=False)
fig.update_layout(margin=dict(l=310))
fig

In [ ]:
ex = sample[sample.decision_challenger == "Decline"].sort_values("pd_challenger", ascending=False).head(3)
for _, r in ex.iterrows():
    rs = [r[f"reason_{i}"] for i in range(1, 5) if r[f"reason_{i}"]]
    sr = [r[f"sc_reason_{i}"] for i in range(1, 5)]
    display(Markdown(f"**Application {int(r.case_id)}** · {r.product_type} · {r.loan_amount:,.0f} CU · challenger PD {r.pd_challenger:.1%} "
                     f"(break-even {r.breakeven_pd:.1%}) → **declined**  \n"
                     f"Challenger reasons: {'; '.join(rs)}  \nScorecard reasons (points lost): {'; '.join(sr)}"))

For each declined applicant we list up to four principal reasons, the inputs that pushed this applicant's risk up the
most (SHAP for the challenger, points below the maximum for the scorecard), using the same plain-English labels. The
dashboard's "Model & Explainability" page shows this for any sampled applicant, with the applicant's value next to the
typical value among approved applicants. None of the reasons can be a protected attribute, because none is an input.

## 4. Limitations

- Hyper-parameters were set once (conservative leaf sizes, early stopping) rather than tuned; a tuning pass would add a
  little Gini but risks overfitting the validation window.
- SHAP explains the model, not the world: "recent rejections" raising the PD is an association, not a cause.
- PDs are calibrated on Nov to Dec 2019 and under-predict the test quarter; absolute PDs should be recalibrated on the
  most recent mature vintage before use.
- Hashed reason codes of earlier rejections cannot be shown to customers as text without the lender's code list.

## 5. Appendix: class imbalance, why we did not resample

In [ ]:
imbalance.round(4)

With 33,000 defaults in the training window, the minority class is not small in absolute terms. Re-weighting and SMOTE
change the *scale* of the output, not the ranking: AUCs stay close, but the raw "probabilities" jump far above the
true default rate. Feed those outputs into the break-even cut-off and the policy declines a large share of good
customers; recalibrating them brings the decision back in line with the unweighted model, so the resampling added
nothing but a calibration step. SMOTE also creates synthetic applicants that never existed, which is hard to defend in
a model review. Decision: train on the true class mix, calibrate, and move the cut-off using costs.

## 6. Appendix: where should the development window start?

Bureau enquiry data was not collected for about 45% of January 2019 applications (6% from June 2019), and several
internal fields started part-way through 2019 (those were dropped in notebook 02). Starting the training window later
gives cleaner data but fewer defaults. The same challenger set-up is refit with later start months:

In [ ]:
win_rows = [{"train_from": "2019-01 (used)", "loans": int(M["train"].sum()), "bads": int(y[M["train"]].sum()),
             "valid_gini": gini(y[va], pd_chal[va]), "test_gini": gini(y[te], pd_chal[te])}]
for start in [201903, 201906]:
    mtr = M["train"] & (df.decision_month >= start).to_numpy()
    Xs_tr, lv_s = prepare_lgbm_frame(df[mtr], FEATURES, CATEGORICAL)
    b_s = train_lgbm(Xs_tr, y[mtr], prepare_lgbm_frame(df[M["valid"]], FEATURES, CATEGORICAL, lv_s)[0], y[M["valid"]], seed=SEED)
    p_s = b_s.predict(prepare_lgbm_frame(df[M["valid"] | M["test"]], FEATURES, CATEGORICAL, lv_s)[0])
    vt = (M["valid"] | M["test"])
    pv = pd.Series(p_s, index=np.nonzero(vt)[0])
    win_rows.append({"train_from": f"{start // 100}-{start % 100:02d}", "loans": int(mtr.sum()), "bads": int(y[mtr].sum()),
                     "valid_gini": gini(y[va], pv.loc[np.nonzero(va)[0]].to_numpy()), "test_gini": gini(y[te], pv.loc[np.nonzero(te)[0]].to_numpy())})
window_check = pd.DataFrame(win_rows)
window_check.to_parquet(OUT / "training_window_check.parquet", index=False)
window_check.round(4)

Starting later costs Gini on both the validation and the test window, and bureau inputs carry only about 5% of the
model's gain, so the full January to October 2019 window is kept. The early-2019 bureau gap stays on the monitoring list.